In [1]:
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from google.colab import drive

drive.mount('/content/gdrive')

df = pd.read_csv('/content/gdrive/My Drive/NCKH/NGHIÊN CỨU KHOA HỌC/Code/P2P_Dataset.csv')
df.head()

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


,acc_now_delinq,acc_open_past_24mths,annual_inc,badloan,chargeoff_within_12_mths,collections_12_mths_ex_med,delinq_2yrs,delinq_amnt,dti,earnings,...,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding
0,0,2,49800.0,0,0,0,0,0,25.120001,995.80,...,False,False,False,False,False,False,False,False,False,False
1,0,3,34000.0,0,0,0,0,0,9.220000,995.80,...,False,False,False,False,False,False,False,False,False,False
2,0,0,34000.0,0,0,0,0,0,25.590000,995.80,...,False,False,False,False,False,False,False,False,False,False
3,0,2,50000.0,1,0,0,0,0,9.310000,750.89,...,False,False,True,False,False,False,False,False,False,False
4,0,2,29000.0,1,0,0,0,0,8.150000,1356.23,...,False,False,False,False,False,False,False,False,False,False


In [2]:
boolean_cols = df.select_dtypes(include=['bool']).columns
df[boolean_cols] = df[boolean_cols].astype(int)
df = df.drop(columns=["badloan", "funded_amnt", "sub_grade", "pub_rec", "num_tl_30dpd"])

In [3]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
from imblearn.over_sampling import SMOTE

# 🎯 Chia X, y
X = df.drop(columns=["default_binary"], errors="ignore")
y = df["default_binary"]

# 🔀 Chia tập train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50, stratify=y)

# ⚖️ Cân bằng dữ liệu bằng SMOTE
smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

from xgboost import XGBClassifier
# 🚀 Huấn luyện mô hình XGBoost cơ bản
xgb_model = XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric="logloss")
xgb_model.fit(X_train_resampled, y_train_resampled)

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [03:53:14] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

In [4]:
selected = ["emp_length", "inq_last_12m", "int_rate", "pct_tl_nvr_dlq", "term", "home_ownership_MORTGAGE",
            "home_ownership_OWN", "home_ownership_RENT", "purpose_credit_card", "purpose_debt_consolidation",
            "purpose_home_improvement", "purpose_house", "purpose_major_purchase", "purpose_medical", "purpose_moving",
            "purpose_other", "purpose_small_business", "purpose_vacation"]
# 🛠 Kiểm tra xem các biến trong `selected` có tồn tại trong tập dữ liệu không
selected_features = [col for col in selected if col in X_train_resampled.columns]

print(f"📌 Các biến thực sự được sử dụng: {selected_features}")

# 🔄 Cập nhật X_train và X_test chỉ với các biến quan trọng
X_train_selected = X_train_resampled[selected_features]
X_test_selected = X_test[selected_features]

# 🔄 Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_train_selected_scaled = scaler.fit_transform(X_train_selected)
X_test_selected_scaled = scaler.transform(X_test_selected)

# 🛠 Kiểm tra kích thước dữ liệu sau khi lọc
print(f"✅ Kích thước X_train_selected_scaled: {X_train_selected_scaled.shape}")
print(f"✅ Kích thước X_test_selected_scaled: {X_test_selected_scaled.shape}")

📌 Các biến thực sự được sử dụng: ['emp_length', 'inq_last_12m', 'int_rate', 'pct_tl_nvr_dlq', 'term', 'home_ownership_MORTGAGE', 'home_ownership_OWN', 'home_ownership_RENT', 'purpose_credit_card', 'purpose_debt_consolidation', 'purpose_home_improvement', 'purpose_house', 'purpose_major_purchase', 'purpose_medical', 'purpose_moving', 'purpose_other', 'purpose_small_business', 'purpose_vacation']
✅ Kích thước X_train_selected_scaled: (2964231, 18)
✅ Kích thước X_test_selected_scaled: (540686, 18)


In [5]:
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

# 🛠 Định nghĩa tập siêu tham số (giới hạn số lượng tổ hợp để tăng tốc)
param_dist = {
    "n_estimators": [100, 300, 500, 700],  # Số cây
    "max_depth": [3, 5, 7, 10],  # Độ sâu
    "learning_rate": [0.01, 0.05, 0.1, 0.2],  # Tốc độ học
    "subsample": [0.5, 0.7, 0.9, 1.0],  # Tỷ lệ mẫu
    "colsample_bytree": [0.5, 0.7, 0.9, 1.0],  # Tỷ lệ cột
    "gamma": [0, 0.1, 0.5, 1, 5],  # Giảm overfitting
    "lambda": [0, 0.1, 1, 2],  # Regularization L2
    "alpha": [0, 0.1, 1, 2]  # Regularization L1
}

# 🚀 RandomizedSearchCV (n_iter=20 để tăng tốc)
random_search = RandomizedSearchCV(
    xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric="logloss"),
    param_distributions=param_dist,
    n_iter=10,  # Chỉ thử 20 tổ hợp thay vì toàn bộ
    scoring="roc_auc",
    cv=2,  # Giảm số lần cross-validation để chạy nhanh hơn
    n_jobs=-1,  # Chạy song song
    random_state=42
)
# 🔥 Huấn luyện mô hình và tìm siêu tham số tối ưu
random_search.fit(X_train_selected_scaled, y_train_resampled)

# 📌 Lấy siêu tham số tốt nhất
best_params = random_search.best_params_
print("\n🔥 Best Hyperparameters for XGBoost:")
print(best_params)

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [04:14:42] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



🔥 Best Hyperparameters for XGBoost:
{'subsample': 0.9, 'n_estimators': 300, 'max_depth': 7, 'learning_rate': 0.05, 'lambda': 1, 'gamma': 1, 'colsample_bytree': 0.9, 'alpha': 1}


In [6]:
# 🏆 Huấn luyện lại mô hình với siêu tham số tối ưu
best_xgb_model = xgb.XGBClassifier(**best_params, random_state=42, use_label_encoder=False, eval_metric="logloss")
best_xgb_model.fit(X_train_selected_scaled, y_train_resampled)

# 📊 Dự đoán trên tập kiểm tra
y_pred_xgb = best_xgb_model.predict(X_test_selected_scaled)
y_prob_xgb = best_xgb_model.predict_proba(X_test_selected_scaled)[:, 1]

# 📈 Đánh giá mô hình
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
roc_auc_xgb = roc_auc_score(y_test, y_prob_xgb)
conf_matrix_xgb = confusion_matrix(y_test, y_pred_xgb)
report_xgb = classification_report(y_test, y_pred_xgb)

# 📝 In kết quả
print("\n XGBoost Model Evaluation (Optimized with RandomizedSearchCV)")
print(f" Accuracy: {accuracy_xgb:.4f}")
print(f" AUC-ROC: {roc_auc_xgb:.4f}")
print(f" Classification Report:\n{report_xgb}")
print(f" Confusion Matrix:\n{conf_matrix_xgb}")

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [04:16:32] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)



 XGBoost Model Evaluation (Optimized with RandomizedSearchCV)
 Accuracy: 0.9095
 AUC-ROC: 0.7280
 Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.99      0.95    494039
           1       0.12      0.01      0.01     46647

    accuracy                           0.91    540686
   macro avg       0.52      0.50      0.48    540686
weighted avg       0.85      0.91      0.87    540686

 Confusion Matrix:
[[491427   2612]
 [ 46297    350]]
